In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql import functions as F
from pyspark.sql.window import Window


In [0]:
spark = SparkSession.builder \
    .appName("HDB Resale Data Load") \
    .getOrCreate()


# File 1
df_approval = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(
        "data/raw/Resale Flat Prices (Based on Approval Date), 2000 - Feb 2012.csv"
    )
    .filter(col("month") >= "2012-01")
)

df_approval.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("hdb_resale_approval_date_012012_022012")


# File 2
df_registration_2012_2014 = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("data/raw/Resale Flat Prices (Based on Registration Date), From Mar 2012 to Dec 2014.csv")

df_registration_2012_2014.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("hdb_resale_registration_date_2012_2014")


# File 3, additionally droping a column remaining_lease as instructed to recompute 
df_registration_2015_2016 = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("data/raw/Resale Flat Prices (Based on Registration Date), From Jan 2015 to Dec 2016.csv")


df_registration_2015_2016.drop("remaining_lease") \
    .write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("hdb_resale_registration_date_2015_2016")

### creating master table with filtered data, also renaming month as a date column

In [0]:
master_df = (
    spark.table("hdb_resale_approval_date_012012_022012")
    .unionByName(
        spark.table("hdb_resale_registration_date_2012_2014"),
        allowMissingColumns=True
    )
    .unionByName(
        spark.table("hdb_resale_registration_date_2015_2016"),
        allowMissingColumns=True
    )
)

# month: 2000-01 -> date: 2000-01-01
master_df = (
    master_df
    .withColumnRenamed("month", "date")
    .withColumn(
        "date",
        F.to_date(F.col("date"), "yyyy-MM-dd")
    )
)

# Update same column:
# 1992 -> 1992-01-01
# 1985 -> 1985-01-01
master_df = master_df.withColumn(
    "lease_commence_date",
    F.concat(
        F.col("lease_commence_date").cast("string"),
        F.lit("-01-01")
    )
)

master_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("hdb_resale_2012_2016")

### Creating Cleaned table by deriving remaining_lease column also considering highest Resale price for given other columns

In [0]:
# Read original table
df = spark.table("hdb_resale_2012_2016")

# Calculate remaining lease in months
remaining_months = F.floor(
    F.months_between(
        F.add_months(F.col("lease_commence_date"), 99 * 12),
        F.col("date")
    )
)

# Create remaining_lease column
df_temp = df.withColumn(
    "remaining_lease",
    F.concat(
        F.floor(remaining_months / 12).cast("int"),
        F.lit(" Years "),
        F.floor(remaining_months % 12).cast("int"),
        F.lit(" Months")
    )
)

# Composite key: all columns except resale_price
composite_key = [
    "date",
    "town",
    "flat_type",
    "block",
    "street_name",
    "storey_range",
    "floor_area_sqm",
    "flat_model",
    "lease_commence_date",
    "remaining_lease"
]

# Rank duplicate keys by highest resale_price
window_spec = Window.partitionBy(*composite_key).orderBy(
    F.col("resale_price").desc()
)

# Keep only the record with the highest resale price
df_temp = (
    df_temp
    .withColumn("row_num", F.row_number().over(window_spec))
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)

# Create temporary view
df_temp.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("hdb_resale_2012_2016_cleaned")

### Transformation 

In [0]:
# 1. Read source table

df = spark.table("hdb_resale_2012_2016_cleaned")


# ============================================================
# 2. Calculate average resale price
#    Group by date, town and flat_type
# ============================================================

avg_price_df = (
    df
    .groupBy("date", "town", "flat_type")
    .agg(
        F.avg("resale_price").alias("avg_resale_price")
    )
)


# Join average resale price back to the original dataset
df = df.join(
    avg_price_df,
    on=["date", "town", "flat_type"],
    how="left"
)


# ============================================================
# 3. Create resale_identifier
#
# Format:
# S + first 3 digits of block + first 2 digits of average price
# + month + first character of town
# ============================================================

# Extract numeric characters from block
block_digits = F.regexp_replace(
    F.col("block"),
    "[^0-9]",
    ""
)

# Take first 3 digits and left-pad with zeroes
block_code = F.lpad(
    F.substring(block_digits, 1, 3),
    3,
    "0"
)

# Convert average resale price to integer and extract first 2 digits
avg_price_code = F.substring(
    F.regexp_replace(
        F.floor(F.col("avg_resale_price")).cast("string"),
        "[^0-9]",
        ""
    ),
    1,
    2
)

# Extract month as two digits
month_code = F.date_format(
    F.to_date("date"),
    "MM"
)

# First character of town
town_code = F.upper(
    F.substring(F.col("town"), 1, 1)
)


df = df.withColumn(
    "resale_identifier",
    F.concat(
        F.lit("S"),
        block_code,
        avg_price_code,
        month_code,
        town_code
    )
)


# ============================================================
# 4. Remove duplicate records
#
# If duplicate resale_identifier exists,
# retain the record with the highest resale_price.
# ============================================================

window_spec = Window.partitionBy(
    "resale_identifier"
).orderBy(
    F.col("resale_price").desc()
)

df = (
    df
    .withColumn(
        "duplicate_rank",
        F.row_number().over(window_spec)
    )
    .filter(
        F.col("duplicate_rank") == 1
    )
    .drop(
        "duplicate_rank",
        "avg_resale_price"
    )
)


# ============================================================
# 5. Hash resale_identifier using SHA-256
#
# SHA-256 is:
# - Irreversible for practical purposes
# - Deterministic
# - Produces a fixed 256-bit hash
# - Suitable for replacing the original identifier
# ============================================================

df = df.withColumn(
    "resale_identifier",
    F.sha2(
        F.col("resale_identifier"),
        256
    )
)


# ============================================================
# 6. Validate uniqueness after hashing
# ============================================================

duplicate_hash_count = (
    df
    .groupBy("resale_identifier")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

if duplicate_hash_count > 0:
    raise Exception(
        "Hash collision detected. Hashed resale_identifier is not unique."
    )


# ============================================================
# 7. Select final schema
# ============================================================

df_final = df.select(
    "date",
    "town",
    "flat_type",
    "block",
    "street_name",
    "storey_range",
    "floor_area_sqm",
    "flat_model",
    "lease_commence_date",
    "remaining_lease",
    "resale_identifier",
    "resale_price"
)


# ============================================================
# 8. Save final cleaned table
# ============================================================

df_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("hdb_resale_2012_2016_final")